In [1]:
%load_ext autoreload
%autoreload 2

In [45]:
from transformers import AutoProcessor, AutoModelForCausalLM
import torch


model_path = "model/florence2"
model = (
    AutoModelForCausalLM.from_pretrained(
        model_path,
        torch_dtype=torch.float16,
        # load_in_4bit=True,
        trust_remote_code=True,
    )
    .cuda()
    .eval()
)
processor = AutoProcessor.from_pretrained(model_path, trust_remote_code=True)


In [41]:
# debug: 卸载模型
del model
del processor
torch.cuda.empty_cache()

In [3]:
img_paths = [
    "/home/zaln/文档/AAvscode/magiv3/input/konan1.jpg",
    "/home/zaln/文档/AAvscode/magiv3/input/konan2.jpg",
    "/home/zaln/文档/AAvscode/magiv3/input/konan3.jpg",
]

In [6]:
# 原始调用

from PIL import Image
from model.florence2.utils import visualise_single_image_prediction


images = [Image.open(img).convert("RGB") for img in img_paths]
results = model.predict_detections_and_associations(images, processor)
for result in results:
    result["dialog_confidences"] = [1.0] * len(result["texts"]) 

import numpy as np
for img_idx in range(len(images)):
    visualise_single_image_prediction(
        np.array(images[img_idx]), results[img_idx], "output/" + "original_" + img_paths[img_idx].split("/")[-1]
    )

# print(result.keys())
# print(result)

/home/zaln/miniconda3/envs/magiv3/lib/python3.10/site-packages/torch/utils/checkpoint.py:86: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(


In [ ]:
# 旧 串联两图 character cluster id
from PIL import Image
from ocr_utils import (
    create_combined_characters_collage,
    predict_character_associations_only,
    visualize_character_associations,
)

images = [Image.open(img).convert("RGB") for img in img_paths]

for idx in range(len(img_paths) - 1):
    co_img, coords1, coords2 = create_combined_characters_collage(
        images[idx],
        images[idx + 1],
        boxes1=results[idx]["characters"],
        boxes2=results[idx + 1]["characters"],
        gap=50,
        save=False,
    )
    
    co_character_associations = predict_character_associations_only(
        model, processor, [co_img]
    )
    visualize_character_associations([co_img], co_character_associations, "./output")


In [9]:
print("res_page1[0]['global_character_ids']:")
print(res_page1[0]["global_character_ids"])
print("res_page2[0]['global_character_ids']:")
print(res_page2[0]["global_character_ids"])


res_page1[0]['global_character_ids']:
[0, 1, 0, 2, 0, 2, 2, 3]
res_page2[0]['global_character_ids']:
[4, 2, 4, 2, 2]


In [10]:
from ocr_utils import visualize_character_associations
visualize_character_associations(images, [res_page1[0], res_page2[0]], "./output")

Visual saved: ./output/0_characters.jpg
Visual saved: ./output/1_characters.jpg


# OCR - paddleocr

In [40]:
# 注入后调用（多张）
global_character_library = []

from ocr_utils import get_ocr_results, predict_with_injected_ocr_and_global_id
from model.florence2.utils import visualise_single_image_prediction

unordered_ocr_res = get_ocr_results(img_paths, only_white_bg=False, zh_texts=True)
results = predict_with_injected_ocr_and_global_id(
    model,
    processor,
    img_paths,
    unordered_ocr_res,
    global_character_library=global_character_library,
    debug=True,
)

# 绘制

import numpy as np
from PIL import Image
from ocr_utils import print_texts


images = [Image.open(img).convert("RGB") for img in img_paths]
for result in results:
    result["dialog_confidences"] = [1.0] * len(result["texts"]) 

import numpy as np
for img_idx in range(len(images)):
    visualise_single_image_prediction(
        np.array(images[img_idx]), results[img_idx], "output/" + "origin_" + img_paths[img_idx].split("/")[-1]
    )
    print_texts(img_paths[img_idx], results[img_idx]["ocr_texts"])

from ocr_utils import visualize_character_associations
visualize_character_associations(images, results, "./output")

/home/zaln/miniconda3/envs/magiv3/lib/python3.10/site-packages/torch/utils/checkpoint.py:86: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(



[DEBUG] Image 0: global library empty → create new IDs
  cluster 0 → new global_id 0
  cluster 1 → new global_id 1
  cluster 2 → new global_id 2
  cluster 3 → new global_id 3
  cluster 4 → new global_id 4
  cluster 5 → new global_id 5
  cluster 6 → new global_id 6
  cluster 7 → new global_id 7
  cluster 8 → new global_id 8
  cluster 9 → new global_id 9

[DEBUG] Image 1
clusters: 8 | library: 10
cluster            0         1         2         3         4         5         6         7         8         9
0             0.8008    0.8193    0.8022    0.6040    0.8916    0.8691    0.7432    0.7256    0.8286    0.8066
1             0.9082    0.9028    0.9102    0.7012    0.8223    0.7437    0.8003    0.6797    0.8237    0.8506
2             0.9121    0.9189    0.9150    0.7085    0.8271    0.7407    0.7764    0.6626    0.8130    0.8491
3             0.8301    0.8438    0.8340    0.6060    0.8882    0.8701    0.7544    0.7188    0.8618    0.8501
4             0.7856    0.7881    0.7705    0.

In [ ]:
for i, panel in enumerate(results[0]["panels"]):
    img.crop(panel).save(f"input/panels/panel{i}_en.jpg")


In [4]:
print(results[0].keys())

results

dict_keys(['panels', 'texts', 'characters', 'tails', 'ocr_texts', 'text_panel_associations', 'character_cluster_labels', 'text_character_associations', 'text_tail_associations', 'is_essential_text', 'dialog_confidences'])


[{'panels': [[395.03997802734375,
    0.7055000066757202,
    919.199951171875,
    487.5005187988281],
   [0.47999998927116394,
    65.61150360107422,
    393.1199951171875,
    487.5005187988281],
   [448.79998779296875, 505.843505859375, 919.199951171875, 851.5385131835938],
   [0.47999998927116394,
    505.843505859375,
    450.7200012207031,
    851.5385131835938],
   [631.2000122070312,
    867.0595092773438,
    919.199951171875,
    1410.2945556640625],
   [375.8399963378906,
    864.2374877929688,
    629.2799682617188,
    1339.7445068359375],
   [0.47999998927116394,
    860.0045166015625,
    377.7599792480469,
    1410.2945556640625]],
  'texts': [[825.0, 50.0, 885.0, 204.0],
   [720.0, 86.0, 804.0, 250.0],
   [458.0, 73.0, 570.0, 281.0],
   [251.0, 73.0, 344.0, 197.0],
   [54.0, 315.0, 194.0, 456.0],
   [875.0, 520.0, 907.0, 651.0],
   [755.0, 555.0, 864.0, 697.0],
   [476.0, 547.0, 537.0, 689.0],
   [329.0, 531.0, 421.0, 659.0],
   [76.0, 520.0, 199.0, 681.0],
   [68.0, 

# caption - llamacpp

In [42]:
from ocr_utils import get_captions
captions = get_captions(img_paths, results, think=False)


In [43]:
for idx, caption in enumerate(captions[0]):
    print(f"🍥 panel{idx}:")
    print(caption)
    print("\n"+"-" * 90 + "\n")


🍥 panel0:
In this black-and-white manga panel, a man dominates the foreground with an intense expression — his eyes wide and eyebrows furrowed in apparent shock or disbelief, mouth slightly open as if mid-speech or reacting to something startling. He wears a knitted beanie and a zippered jacket, suggesting cold weather. Behind him, slightly out of focus, stands another man holding what appears to be popcorn; he is dressed warmly in a puffy coat and hat, looking toward the foreground figure with a neutral or concerned expression. To the left and partially obscured by the main character’s head, a third person — possibly female, given the silhouette and hat — holds a smartphone, seemingly recording or observing the scene. The background features dynamic speed lines radiating outward, emphasizing tension or sudden movement. All characters are rendered in high-contrast ink work typical of Japanese comics, with no visible text or speech bubbles included in the visual description.

----------

# character grounding

In [46]:
from PIL import Image

grounded_results = []
for i in range(len(img_paths)):
    # 第 i 张图片的全部 panel ，串行
    grounded_result = []
    img = Image.open(img_paths[i])
    img_panels = [img.crop(panel) for panel in results[i]["panels"]]
    for img, cap in zip(img_panels, captions[i]):
        res = model.predict_character_grounding(
            [img],
            [cap],
            processor,
            strict=False,
        )
        grounded_result.extend(res)
    grounded_results.append(grounded_result)


/home/zaln/miniconda3/envs/magiv3/lib/python3.10/site-packages/torch/utils/checkpoint.py:86: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(


# character cluster id 插入 caption

In [47]:
from ocr_utils import preprocess_panel_characters, get_grounding

# 针对当前页进行预处理，获得按 panel 划分并转换好局部坐标的 characters 列表

grounded_captions = []
# 遍历每张图片
for img_idx in range(len(img_paths)):
    panel_characters_list = preprocess_panel_characters(results[img_idx])
    grounded_result = grounded_results[img_idx]
    grounded_caption = []
    # 遍历每个 panel
    for panel_idx, res in enumerate(grounded_result):
        # 为当前 panel 分配 characters
        panel_characters = panel_characters_list[panel_idx]
        
        # 无可视化
        cap = get_grounding(res, panel_characters, None)

        # # 可视化
        # panel_img = (
        #     Image.open(img_paths[img_idx])
        #     .crop(results[img_idx]["panels"][panel_idx])
        #     .convert("RGB")
        # )
        # cap = get_grounding(res, panel_characters, panel_img)

        grounded_caption.append(cap)
    grounded_captions.append(grounded_caption)

In [48]:
grounded_captions

[['In this black-and-white manga panel, a man[0] dominates the foreground with an intense expression — his[0] eyes wide and eyebrows furrowed in apparent shock or disbelief, mouth slightly open as if mid-speech or reacting to something startling. He[0] wears a knitted beanie and a zippered jacket, suggesting cold weather. Behind him[0], slightly out of focus, stands another man[1] holding what appears to be popcorn; he[1] is dressed warmly in a puffy coat and hat, looking toward the foreground figure with a neutral or concerned expression. To the left and partially obscured by the main character’s head, a third person — possibly female, given the silhouette and hat — holds a smartphone, seemingly recording or observing the scene. The background features dynamic speed lines radiating outward, emphasizing tension or sudden movement. All characters are rendered in high-contrast ink work typical of Japanese comics, with no visible text or speech bubbles included in the visual description.'

# 最终话

## 1. 构建 panel_scripts

In [77]:
from ocr_utils import build_panel_scripts

panel_scripts = []
# List of list of "one panel script"
for result in results:
    panel_scripts.append(build_panel_scripts(result, essential_only=False, include_narrator=True, label="narrator", label_char_name="character"))


In [78]:
# 打印结果
for i, panel_script in enumerate(panel_scripts):
    print(f"====== img {i} ======")
    for j, script in enumerate(panel_script):
        print(f"===== panel {j} =====")
        for line in script:
            print(line)
        print()

====== img 0 ======
===== panel 0 =====
narrator: “我们4个人看电影的时候一直是我去买爆米花的……”
character 0: “吉他手明石塔介(30)”
character 0: “小萤和初音的份都是我一起买的……”

===== panel 1 =====
character 3: “除了爆米花之外被害人还拿了什么东西吗……？”
character 5: “我我也有交给她的！”

===== panel 2 =====
character 6: “我在这里的商店里买的周边…”
character 5: “贝斯手堂村萤(27)”
character 5: “一边还和她说着很可爱吧？”

===== panel 3 =====
character 3: “你不只是拿出来给她看还交给她了是吧？”
character 5: “是是的！是我们在演唱会的中途会扔给现场观众作为礼物的小东西”
narrator: “我希望她能亲自确认一下手感……”

===== panel 4 =====
narrator: “然后我给所有人买来了饮料……”
character 6: “鼓手贤木光男(29)”
character 6: “初音一直是要无糖的冰茶’我亲手递给她的……”

===== panel 5 =====
character 8: “直点的都是样的吗？”
character 6: “是啊！塔介是冰咖啡！小萤是可乐’我是乌龙茶……”
character 9: “这样的话等鉴定科来调查一下毒是用什么方法下到什么东西里面的话：…”
character 9: “马上就知道犯人是谁了吧？”

===== panel 6 =====
character 9: “就在被害人身边的你们三个嫌疑人……”
character 9: “就算想把毒药藏起来也是做不到的！”
character 9: “因为案发之后你们一步也没有离开过这里……”

====== img 1 ======
===== panel 0 =====
character 4: “你说什么”
character 2: “兴”
narrator: “天文馆杯户”
narrator: “没有任何地方沾到了毒药？”

===== panel 1 =====
character 5: “

## 2. 构建 prose_prompt

In [83]:
from ocr_utils import get_prose_prompt

prose_prompt = get_prose_prompt(grounded_captions, panel_scripts)

In [84]:
for line in prose_prompt:
    print(line)

I have a series of manga panel descriptions and dialogues.

Panel 1

Description:
In this black-and-white manga panel, a man[0] dominates the foreground with an intense expression — his[0] eyes wide and eyebrows furrowed in apparent shock or disbelief, mouth slightly open as if mid-speech or reacting to something startling. He[0] wears a knitted beanie and a zippered jacket, suggesting cold weather. Behind him[0], slightly out of focus, stands another man[1] holding what appears to be popcorn; he[1] is dressed warmly in a puffy coat and hat, looking toward the foreground figure with a neutral or concerned expression. To the left and partially obscured by the main character’s head, a third person — possibly female, given the silhouette and hat — holds a smartphone, seemingly recording or observing the scene. The background features dynamic speed lines radiating outward, emphasizing tension or sudden movement. All characters are rendered in high-contrast ink work typical of Japanese comi